In [4]:
import tensorflow as tf
import sys
import matplotlib.pyplot as plt
import numpy as np

sys.path.append('../../datasets')
from load_data import get_datasets

from model import get_baseline_model

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
sys.path.append("./../datasets/src")
from src.dataloader_daliac import DaliacDataLoader

dl = DaliacDataLoader(data_path="../../datasets/daliac/", return_one_hot=True)

ds_train, ds_val, ds_test = dl.load_complete_dataset()

In [38]:
#ds_train, ds_val, ds_test, class_weights = get_datasets("daliac", "../../datasets/daliac/", return_one_hot=True)

labels = {1: 'sitting', 2: 'lying', 3: 'standing', 4: 'washing_dishes', 5: 'vacuuming', 
          6: 'sweeping',  7: 'walking', 8: 'ascending_stairs', 9: 'descending_stairs', 10: 'treadmill_running', 
          11: 'bicycling_on_ergometer_50_W', 12: 'bicycling_on_ergometer_100_W', 13: 'rope_jumping'}


In [39]:
# get simple cnn with (2048, 24, 1) input shape (conv2d)
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(2048, 24, 1), padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(13, activation='softmax')
])


model.compile(optimizer=tf.keras.optimizers.Adam(0.0001),
                loss='categorical_crossentropy',
                metrics=['accuracy'])

model.summary()

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_15 (Conv2D)          (None, 2046, 22, 32)      320       
                                                                 
 max_pooling2d_10 (MaxPoolin  (None, 1023, 11, 32)     0         
 g2D)                                                            
                                                                 
 conv2d_16 (Conv2D)          (None, 1021, 9, 64)       18496     
                                                                 
 max_pooling2d_11 (MaxPoolin  (None, 510, 4, 64)       0         
 g2D)                                                            
                                                                 
 conv2d_17 (Conv2D)          (None, 508, 2, 64)        36928     
                                                                 
 flatten_7 (Flatten)         (None, 65024)            

In [40]:
model.fit(ds_train.batch(128), validation_data=ds_val.batch(128), epochs=10)

Epoch 1/10


26/26 [==============================] - 19s 709ms/step - loss: 1.9891 - accuracy: 0.3384 - val_loss: 1.5605 - val_accuracy: 0.5081
Epoch 2/10
26/26 [==============================] - 18s 704ms/step - loss: 1.4664 - accuracy: 0.5314 - val_loss: 1.4424 - val_accuracy: 0.5360
Epoch 3/10
26/26 [==============================] - 18s 707ms/step - loss: 1.1996 - accuracy: 0.6192 - val_loss: 1.2510 - val_accuracy: 0.6125
Epoch 4/10
26/26 [==============================] - 18s 708ms/step - loss: 0.9914 - accuracy: 0.6715 - val_loss: 1.1013 - val_accuracy: 0.6589
Epoch 5/10
26/26 [==============================] - 18s 704ms/step - loss: 0.8178 - accuracy: 0.7281 - val_loss: 1.0497 - val_accuracy: 0.7239
Epoch 6/10
26/26 [==============================] - 18s 709ms/step - loss: 0.6844 - accuracy: 0.7785 - val_loss: 1.0434 - val_accuracy: 0.7262
Epoch 7/10
26/26 [==============================] - 18s 707ms/step - loss: 0.5983 - accuracy: 0.8101 - val_loss: 1.0377 - val_accuracy: 0.7262
Epoch 8/10

In [7]:
model = get_baseline_model(input_length=2048, n_fft=256, hop_length=128, n_mel=128)
model.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 2048, 1)]    0           []                               
                                                                                                  
 stft_2 (STFT)                  (None, 15, 129, 1)   0           ['input_3[0][0]']                
                                                                                                  
 magnitude_2 (Magnitude)        (None, 15, 129, 1)   0           ['stft_2[0][0]']                 
                                                                                                  
 magnitude_to_decibel_2 (Magnit  (None, 15, 129, 1)  0           ['magnitude_2[0][0]']            
 udeToDecibel)                                                                              

In [8]:
model.fit(ds_train.batch(64), validation_data=ds_val.batch(64), epochs=50)

Epoch 1/50


51/51 [==============================] - 4s 35ms/step - loss: 5.0029 - accuracy: 0.2611 - val_loss: 40587341824.0000 - val_accuracy: 0.0139
Epoch 2/50
51/51 [==============================] - 1s 28ms/step - loss: 1.8806 - accuracy: 0.3573 - val_loss: 2.6343 - val_accuracy: 0.0325
Epoch 3/50
51/51 [==============================] - 1s 28ms/step - loss: 1.7004 - accuracy: 0.4367 - val_loss: 708.2209 - val_accuracy: 0.0394
Epoch 4/50
51/51 [==============================] - 1s 28ms/step - loss: 1.5485 - accuracy: 0.4640 - val_loss: 36.1401 - val_accuracy: 0.1183
Epoch 5/50
51/51 [==============================] - 1s 28ms/step - loss: 1.4840 - accuracy: 0.4909 - val_loss: 6.4938 - val_accuracy: 0.2506
Epoch 6/50
51/51 [==============================] - 1s 28ms/step - loss: 1.3344 - accuracy: 0.5373 - val_loss: 2.2497 - val_accuracy: 0.3573
Epoch 7/50
51/51 [==============================] - 1s 28ms/step - loss: 1.4909 - accuracy: 0.4853 - val_loss: 2.4026 - val_accuracy: 0.0742
Epoch 8/50


In [9]:
model.evaluate(ds_test.batch(64))

7/7 [==============================] - 0s 43ms/step - loss: 2.6397 - accuracy: 0.1975


[2.6397361755371094, 0.1974683552980423]